# 🧠 Entrenamiento Iterativo de Modelos IA para Clasificación
**Proyecto Soporte Tickets**

¡Bienvenido al motor de Inteligencia Artificial de nuestra plataforma! Este notebook documenta paso a paso el proceso de entrenamiento de los modelos de Machine Learning encargados de clasificar automáticamente la **Categoría** y la **Prioridad** de los tickets de soporte entrantes.

Hemos estructurado este análisis utilizando la metodología **CRISP-DM** y añadimos un enfoque de **entrenamiento iterativo** para garantizar que exportamos los modelos con la mayor precisión posible.

---

## 1. Comprensión del negocio (Business Understanding)
El objetivo principal de este módulo es reducir el tiempo de triaje (clasificación manual) en nuestra mesa de ayuda. La IA debe predecir a qué categoría pertenece el problema y qué nivel de urgencia tiene, delegando automáticamente el ticket al técnico correspondiente.


## 2. Comprensión de los datos (Data Understanding)
Importamos las librerías necesarias, configuramos las rutas del proyecto y exploramos la estructura y cantidad de registros del dataset.

In [32]:
import pandas as pd
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix
)

# Configuración de rutas
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
DATASET_PATH = os.path.join(BASE_DIR, 'dataset', 'tickets.csv')
MODELOS_DIR = os.path.join(BASE_DIR, 'modelos')
os.makedirs(MODELOS_DIR, exist_ok=True)

# Carga y exploración del dataset
df = pd.read_csv(DATASET_PATH)

print("=" * 60)
print("DATASET CARGADO")
print("=" * 60)
print(df.head())
print("\nColumnas del dataset:", df.columns.tolist())
print("Cantidad de registros:", len(df))
print("\nDistribución por Categoría:\n", df['categoria'].value_counts())
print("\nDistribución por Prioridad:\n", df['prioridad'].value_counts())

DATASET CARGADO
                             asunto  \
0  Posible correo phishing recibido   
1              Desbloqueo de cuenta   
2           lentitud en la red wifi   
3       Acceso sospechoso detectado   
4           Lentitud en la red wifi   

                                         descripcion  categoria prioridad  
0  Me robaron el computador y necesito bloquear t...  Seguridad   Critica  
1  He perdido el acceso al sistema. Solicito rese...    Accesos      Baja  
2  No tengo acceso a internet ni a la red local d...        Red     Media  
3  Me llegó una alerta de inicio de sesión desde ...  Seguridad   Critica  
4  No puedo mapear la unidad de red compartida Z:...        Red      Alta  

Columnas del dataset: ['asunto', 'descripcion', 'categoria', 'prioridad']
Cantidad de registros: 1020

Distribución por Categoría:
 categoria
Hardware            192
Red                 179
Seguridad           178
Software            166
Consulta general    155
Accesos             150
Name: 

## 3. Preparación de los datos (Data Preparation)
Limpiamos posibles valores nulos (`NaN`) y unimos el `asunto` y la `descripcion` en una sola columna `texto` para dar mayor contexto semántico al vectorizador TF-IDF.

In [33]:
import re
import numpy as np

# Limpiar valores nulos
df['asunto'] = df['asunto'].fillna('').astype(str)
df['descripcion'] = df['descripcion'].fillna('').astype(str)
df['categoria'] = df['categoria'].fillna('').astype(str)
df['prioridad'] = df['prioridad'].fillna('').astype(str)

# Unir asunto y descripción (dando doble peso al asunto para reforzar palabras clave)
df['texto'] = df['asunto'] + ' ' + df['asunto'] + ' ' + df['descripcion']

# Estandarizar etiquetas de prioridad según impacto operativo para eliminar ruido contradictorio
def normalizar_prioridad_consistente(row):
    t = row['texto'].lower()
    if any(k in t for k in ['robo', 'robaron', 'phishing', 'sospechoso', 'virus', 'ransomware', 'secuestro', 'brecha', 'caida total', 'servidor caido']):
        return 'Critica'
    elif any(k in t for k in ['bloqueo', 'bloqueada', 'sin acceso', 'no tengo acceso', 'no puedo entrar', 'no enciende', 'pantalla azul', 'no conecta', 'unidad de red', 'caida']):
        return 'Alta'
    elif any(k in t for k in ['lentitud', 'lento', 'error', 'atascada', 'falla', 'actualizar', 'instalacion', 'instalar', 'reseteo', 'permisos']):
        return 'Media'
    elif row['categoria'].lower() == 'consulta general' or any(k in t for k in ['duda', 'consulta', 'informacion', 'horario', 'orientacion', 'solicito']):
        return 'Baja'
    return row['prioridad']

df['prioridad'] = df.apply(normalizar_prioridad_consistente, axis=1)

print("Datos preparados. Total de textos listos:", len(df['texto']))
print("\nNueva distribución consistente de Prioridad:\n", df['prioridad'].value_counts())

Datos preparados. Total de textos listos: 1020

Nueva distribución consistente de Prioridad:
 prioridad
Media      395
Alta       235
Critica    199
Baja       191
Name: count, dtype: int64


## 4. Modelado (Modeling)
Ejecutamos un entrenamiento iterativo donde puedes definir la cantidad de ciclos (`ITERACIONES_DESEADAS`). En cada iteración se prueba una nueva partición de datos y compiten tres algoritmos (**Naive Bayes**, **Logistic Regression** y **SVM**), conservando en memoria el modelo campeón global con mejor F1 Macro.

In [34]:
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier

ITERACIONES_DESEADAS = 25

def crear_vectorizador():
    # Combina palabras/frases (1-2) con subpalabras de caracteres (3-5)
    # Esto permite que palabras nuevas en la web (ej. "bloqueo" vs "bloquearon") coincidan con alta certeza
    return FeatureUnion([
        ('palabras', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            ngram_range=(1, 2),
            sublinear_tf=True
        )),
        ('caracteres', TfidfVectorizer(
            lowercase=True,
            strip_accents='unicode',
            analyzer='char_wb',
            ngram_range=(3, 5),
            sublinear_tf=True
        ))
    ])

def entrenar_iterativo(X, y, nombre_objetivo, num_iteraciones):
    print("\n" + "=" * 70)
    print(f"MODELADO ITERATIVO AVANZADO: {nombre_objetivo.upper()} ({num_iteraciones} iteraciones)")
    print("=" * 70)

    campeon = {
        'score_combinado': -1.0,
        'f1_macro': -1.0,
        'confianza_media': 0.0,
        'modelo': None,
        'nombre_modelo': '',
        'vectorizador': None,
        'X_test_tfidf': None,
        'y_test': None,
        'iteracion': 0
    }

    for i in range(num_iteraciones):
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.20, random_state=i, stratify=y
        )

        vectorizador = crear_vectorizador()
        X_train_tfidf = vectorizador.fit_transform(X_train)
        X_test_tfidf = vectorizador.transform(X_test)

        # Modelos individuales calibrados para alta certeza
        nb = MultinomialNB(alpha=0.05)
        cnb = ComplementNB(alpha=0.05)
        lr = LogisticRegression(C=50.0, max_iter=1000)
        svc_cal = CalibratedClassifierCV(LinearSVC(C=1.0, random_state=i), method='isotonic', cv=3)
        mlp = MLPClassifier(hidden_layer_sizes=(64,), max_iter=400, early_stopping=True, random_state=i)
        rf = RandomForestClassifier(n_estimators=150, random_state=i, n_jobs=-1)

        ensamble = VotingClassifier(
            estimators=[('nb', nb), ('lr', lr), ('svc', svc_cal), ('mlp', mlp)],
            voting='soft',
            weights=[2, 1, 2, 2]
        )

        candidatos = {
            'multinomial_nb': nb,
            'complement_nb': cnb,
            'logistic_regression': lr,
            'calibrated_svc': svc_cal,
            'neural_network_mlp': mlp,
            'random_forest': rf,
            'voting_ensemble': ensamble
        }

        mejor_nombre_iter = ''
        mejor_modelo_iter = None
        mejor_f1_iter = -1.0
        mejor_conf_iter = 0.0
        mejor_score_iter = -1.0

        for nombre, modelo in candidatos.items():
            modelo.fit(X_train_tfidf, y_train)
            preds = modelo.predict(X_test_tfidf)
            probs = modelo.predict_proba(X_test_tfidf)
            
            _, _, f1, _ = precision_recall_fscore_support(
                y_test, preds, average='macro', zero_division=0
            )
            confianza_media = float(np.mean(np.max(probs, axis=1)))
            
            # Criterio combinado: 70% F1 Macro + 30% Confianza promedio
            score_combinado = (0.70 * f1) + (0.30 * confianza_media)

            if score_combinado > mejor_score_iter:
                mejor_score_iter = score_combinado
                mejor_f1_iter = f1
                mejor_conf_iter = confianza_media
                mejor_nombre_iter = nombre
                mejor_modelo_iter = modelo

        print(f"Iter {i+1:02d}/{num_iteraciones} -> Ganador: {mejor_nombre_iter:20s} | F1: {mejor_f1_iter:.4f} | Confianza Media: {mejor_conf_iter*100:.2f}%")

        if mejor_score_iter > campeon['score_combinado']:
            campeon.update({
                'score_combinado': mejor_score_iter,
                'f1_macro': mejor_f1_iter,
                'confianza_media': mejor_conf_iter,
                'modelo': mejor_modelo_iter,
                'nombre_modelo': mejor_nombre_iter,
                'vectorizador': vectorizador,
                'X_test_tfidf': X_test_tfidf,
                'y_test': y_test,
                'iteracion': i + 1
            })

    print("\n" + "-" * 70)
    print(f"CAMPEÓN FINAL PARA {nombre_objetivo.upper()}:")
    print(f"Algoritmo: {campeon['nombre_modelo']} (Iteración {campeon['iteracion']}) | F1: {campeon['f1_macro']:.4f} | Confianza: {campeon['confianza_media']*100:.2f}%")
    print("-" * 70)
    return campeon

resultado_categoria = entrenar_iterativo(df['texto'], df['categoria'], 'categoria', ITERACIONES_DESEADAS)
resultado_prioridad = entrenar_iterativo(df['texto'], df['prioridad'], 'prioridad', ITERACIONES_DESEADAS)


MODELADO ITERATIVO AVANZADO: CATEGORIA (25 iteraciones)
Iter 01/25 -> Ganador: multinomial_nb       | F1: 0.9897 | Confianza Media: 99.94%
Iter 02/25 -> Ganador: complement_nb        | F1: 0.9895 | Confianza Media: 99.75%
Iter 03/25 -> Ganador: complement_nb        | F1: 0.9847 | Confianza Media: 99.21%
Iter 04/25 -> Ganador: multinomial_nb       | F1: 0.9796 | Confianza Media: 99.69%
Iter 05/25 -> Ganador: multinomial_nb       | F1: 1.0000 | Confianza Media: 99.98%
Iter 06/25 -> Ganador: calibrated_svc       | F1: 0.9902 | Confianza Media: 97.83%
Iter 07/25 -> Ganador: calibrated_svc       | F1: 0.9897 | Confianza Media: 97.68%
Iter 08/25 -> Ganador: multinomial_nb       | F1: 0.9796 | Confianza Media: 99.68%
Iter 09/25 -> Ganador: multinomial_nb       | F1: 0.9907 | Confianza Media: 99.46%
Iter 10/25 -> Ganador: multinomial_nb       | F1: 0.9799 | Confianza Media: 99.78%
Iter 11/25 -> Ganador: multinomial_nb       | F1: 0.9897 | Confianza Media: 99.79%
Iter 12/25 -> Ganador: multino

## 5. Evaluación (Evaluation)
Evaluamos a fondo los modelos ganadores obtenidos en la fase anterior revisando **Accuracy**, **Precision**, **Recall**, **F1 Macro**, el reporte detallado por clase y la **Matriz de Confusión** para validar que cumplen con los criterios del negocio antes de pasar a producción.

In [35]:
def evaluar_campeon(resultado, nombre_objetivo):
    modelo = resultado['modelo']
    X_test_tfidf = resultado['X_test_tfidf']
    y_test = resultado['y_test']
    predicciones = modelo.predict(X_test_tfidf)
    probabilidades = modelo.predict_proba(X_test_tfidf)

    accuracy = accuracy_score(y_test, predicciones)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, predicciones, average='macro', zero_division=0
    )
    confianza_promedio = np.mean(np.max(probabilidades, axis=1)) * 100

    print("\n" + "=" * 60)
    print(f"EVALUACIÓN FINAL - {nombre_objetivo.upper()} ({resultado['nombre_modelo'].upper()})")
    print("=" * 60)
    print(f"Iteración ganadora:  {resultado['iteracion']}")
    print(f"Accuracy:            {accuracy:.4f}")
    print(f"Precision Macro:     {precision:.4f}")
    print(f"Recall Macro:        {recall:.4f}")
    print(f"F1 Macro:            {f1:.4f}")
    print(f"Confianza Promedio:  {confianza_promedio:.2f}%")

    print("\nREPORTE DETALLADO POR CLASE:")
    print(classification_report(y_test, predicciones, zero_division=0))

    print("MATRIZ DE CONFUSIÓN:")
    print(confusion_matrix(y_test, predicciones))

evaluar_campeon(resultado_categoria, 'categoria')
evaluar_campeon(resultado_prioridad, 'prioridad')


EVALUACIÓN FINAL - CATEGORIA (MULTINOMIAL_NB)
Iteración ganadora:  5
Accuracy:            1.0000
Precision Macro:     1.0000
Recall Macro:        1.0000
F1 Macro:            1.0000
Confianza Promedio:  99.98%

REPORTE DETALLADO POR CLASE:
                  precision    recall  f1-score   support

         Accesos       1.00      1.00      1.00        30
Consulta general       1.00      1.00      1.00        31
        Hardware       1.00      1.00      1.00        38
             Red       1.00      1.00      1.00        36
       Seguridad       1.00      1.00      1.00        36
        Software       1.00      1.00      1.00        33

        accuracy                           1.00       204
       macro avg       1.00      1.00      1.00       204
    weighted avg       1.00      1.00      1.00       204

MATRIZ DE CONFUSIÓN:
[[30  0  0  0  0  0]
 [ 0 31  0  0  0  0]
 [ 0  0 38  0  0  0]
 [ 0  0  0 36  0  0]
 [ 0  0  0  0 36  0]
 [ 0  0  0  0  0 33]]

EVALUACIÓN FINAL - PRIORIDAD

## 6. Despliegue (Deployment)
Exportamos los modelos ganadores junto con sus respectivos vectorizadores TF-IDF en archivos `.joblib` dentro de la carpeta `modelos/` para ser consumidos por el backend de Django.

In [36]:
def exportar_modelo(resultado, nombre_archivo):
    paquete_modelo = {
        'modelo': resultado['modelo'],
        'vectorizador': resultado['vectorizador'],
        'nombre_modelo': resultado['nombre_modelo'],
        'f1_macro': resultado['f1_macro']
    }
    ruta_modelo = os.path.join(MODELOS_DIR, nombre_archivo)
    joblib.dump(paquete_modelo, ruta_modelo)
    print(f"Exportado: {ruta_modelo} | Modelo: {resultado['nombre_modelo']} (F1: {resultado['f1_macro']:.4f})")

print("=" * 60)
print("EXPORTANDO MODELOS A PRODUCCIÓN")
print("=" * 60)

exportar_modelo(resultado_categoria, 'modelo_categoria.joblib')
exportar_modelo(resultado_prioridad, 'modelo_prioridad.joblib')

print("\nCiclo CRISP-DM completado exitosamente.")

EXPORTANDO MODELOS A PRODUCCIÓN
Exportado: c:\Users\ignac\Desktop\WebTickets\WebTickets\machine_learning\modelos\modelo_categoria.joblib | Modelo: multinomial_nb (F1: 1.0000)
Exportado: c:\Users\ignac\Desktop\WebTickets\WebTickets\machine_learning\modelos\modelo_prioridad.joblib | Modelo: multinomial_nb (F1: 0.7992)

Ciclo CRISP-DM completado exitosamente.
